# Xenium and Phenocycler Data Integration

Integrate Xenium spatial transcriptomics with Phenocycler imaging data from raw .qptiff format:
- Read raw Phenocycler data from .qptiff files
- Account for magnification differences (Xenium 40x vs Phenocycler 20x)
- Resize Xenium segmentation to match Phenocycler scale
- Register resized segmentation to Phenocycler image data
- Perform multi-modal integration

**Input:** Xenium AnnData and Phenocycler .qptiff files

**Output:** Integrated multi-modal dataset with registered spatial coordinates

In [ ]:
import numpy as np
import pandas as pd
import scanpy as sc
import squidpy as sq
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Image processing
import tifffile
from skimage import transform, exposure, filters
from skimage.registration import phase_cross_correlation
from scipy import ndimage

sc.settings.set_figure_params(dpi=80)
print("Libraries loaded successfully")

## 1. Load Xenium Data

In [ ]:
# Define paths
DATA_DIR = Path("../data/processed")
PHENOCYCLER_DIR = Path("../data/phenocycler")
OUTPUT_DIR = Path("../data/processed")
FIGURES_DIR = Path("../figures/06_integration")
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
PHENOCYCLER_DIR.mkdir(parents=True, exist_ok=True)

SAMPLE_NAME = "xenium_sample_01"

# Magnification parameters
XENIUM_MAGNIFICATION = 40  # 40x objective
PHENOCYCLER_MAGNIFICATION = 20  # 20x objective
SCALE_FACTOR = PHENOCYCLER_MAGNIFICATION / XENIUM_MAGNIFICATION  # 0.5

print(f"Magnification scale factor: {SCALE_FACTOR}")
print(f"Xenium data will be scaled by {SCALE_FACTOR}x to match Phenocycler resolution")

# Load Xenium data
try:
    adata_xenium = sc.read_h5ad(DATA_DIR / f"{SAMPLE_NAME}_annotated.h5ad")
    print(f"\nXenium data loaded: {adata_xenium.shape}")
    print(f"Cell types: {adata_xenium.obs['celltype'].nunique()}")
    
    # Check if spatial coordinates exist
    if 'spatial' in adata_xenium.obsm:
        print(f"Spatial coordinates available: {adata_xenium.obsm['spatial'].shape}")
    else:
        print("WARNING: No spatial coordinates found in Xenium data")
except FileNotFoundError:
    print(f"ERROR: Xenium data file not found at {DATA_DIR / f'{SAMPLE_NAME}_annotated.h5ad'}")
    print("Please run notebooks 01-02 first to generate annotated data.")
    adata_xenium = None

## 2. Read Phenocycler .qptiff Data

Load raw Phenocycler imaging data from .qptiff format (pyramidal multi-channel TIFF).

In [ ]:
def read_qptiff(qptiff_path, level=0):
    """
    Read Phenocycler .qptiff file.
    
    Parameters:
    -----------
    qptiff_path : str or Path
        Path to .qptiff file
    level : int
        Pyramid level to read (0 = highest resolution)
    
    Returns:
    --------
    image : ndarray
        Image array with shape (channels, height, width)
    metadata : dict
        Image metadata including channel names
    """
    with tifffile.TiffFile(qptiff_path) as tif:
        # Read the specified pyramid level
        if hasattr(tif, 'series'):
            # Pyramidal TIFF
            if level < len(tif.series[0].levels):
                image = tif.series[0].levels[level].asarray()
            else:
                print(f"Level {level} not found, using level 0")
                image = tif.series[0].levels[0].asarray()
        else:
            # Regular multi-page TIFF
            image = tif.asarray()
        
        # Extract metadata
        metadata = {}
        if tif.pages:
            page = tif.pages[0]
            metadata['shape'] = image.shape
            metadata['dtype'] = image.dtype
            
            # Try to extract channel names from description
            if hasattr(page, 'description') and page.description:
                metadata['description'] = page.description
            
            # Extract resolution if available
            if hasattr(page, 'tags'):
                if 'XResolution' in page.tags:
                    metadata['x_resolution'] = page.tags['XResolution'].value
                if 'YResolution' in page.tags:
                    metadata['y_resolution'] = page.tags['YResolution'].value
    
    return image, metadata


def create_composite_image(image, channels_to_use=None, channel_colors=None):
    """
    Create RGB composite from multi-channel image.
    
    Parameters:
    -----------
    image : ndarray
        Multi-channel image (channels, height, width)
    channels_to_use : list of int, optional
        Indices of channels to use (max 3 for RGB)
    channel_colors : list of str, optional
        Colors for each channel ('red', 'green', 'blue')
    
    Returns:
    --------
    composite : ndarray
        RGB composite image (height, width, 3)
    """
    if image.ndim == 2:
        # Single channel image
        composite = np.stack([image, image, image], axis=-1)
    elif image.ndim == 3:
        # Multi-channel image
        n_channels = image.shape[0]
        
        if channels_to_use is None:
            # Use first 3 channels
            channels_to_use = list(range(min(3, n_channels)))
        
        # Create RGB composite
        h, w = image.shape[1:]
        composite = np.zeros((h, w, 3), dtype=np.float32)
        
        for i, ch_idx in enumerate(channels_to_use[:3]):
            if ch_idx < n_channels:
                # Normalize channel to 0-1 range
                ch_data = image[ch_idx].astype(np.float32)
                ch_data = (ch_data - ch_data.min()) / (ch_data.max() - ch_data.min() + 1e-8)
                composite[:, :, i] = ch_data
    else:
        raise ValueError(f"Unexpected image dimensions: {image.shape}")
    
    return composite


# Load Phenocycler data
QPTIFF_FILE = PHENOCYCLER_DIR / f"{SAMPLE_NAME}.qptiff"

if QPTIFF_FILE.exists():
    print(f"Loading Phenocycler data from: {QPTIFF_FILE}")
    phenocycler_image, pheno_metadata = read_qptiff(QPTIFF_FILE, level=0)
    
    print(f"\nPhenocycler image loaded:")
    print(f"  Shape: {phenocycler_image.shape}")
    print(f"  Dtype: {phenocycler_image.dtype}")
    print(f"  Range: [{phenocycler_image.min()}, {phenocycler_image.max()}]")
    
    # Determine image organization
    if phenocycler_image.ndim == 3:
        n_channels = phenocycler_image.shape[0]
        img_height = phenocycler_image.shape[1]
        img_width = phenocycler_image.shape[2]
        print(f"  Channels: {n_channels}")
        print(f"  Dimensions: {img_height} x {img_width}")
    elif phenocycler_image.ndim == 2:
        print(f"  Single channel image: {phenocycler_image.shape}")
        img_height, img_width = phenocycler_image.shape
    
    # Create and display composite
    composite = create_composite_image(phenocycler_image)
    
    plt.figure(figsize=(12, 10))
    plt.imshow(composite)
    plt.title('Phenocycler Composite Image (20x magnification)')
    plt.axis('off')
    plt.tight_layout()
    plt.savefig(FIGURES_DIR / 'phenocycler_composite.png', dpi=300, bbox_inches='tight')
    plt.show()
    
else:
    print(f"\nPhenocycler .qptiff file not found at: {QPTIFF_FILE}")
    print("Please place your Phenocycler .qptiff file in the data/phenocycler/ directory.")
    print("\nExpected filename format: {SAMPLE_NAME}.qptiff")
    print(f"Example: {SAMPLE_NAME}.qptiff")
    phenocycler_image = None

## 3. Create Xenium Segmentation Mask

Convert Xenium cell spatial coordinates to a segmentation mask.

In [ ]:
def create_segmentation_from_coords(adata, output_shape=None, cell_radius=5):
    """
    Create segmentation mask from cell coordinates.
    
    Parameters:
    -----------
    adata : AnnData
        Annotated data with spatial coordinates in obsm['spatial']
    output_shape : tuple, optional
        Output image shape (height, width). If None, calculated from coordinates.
    cell_radius : int
        Radius of each cell in pixels
    
    Returns:
    --------
    segmentation : ndarray
        Segmentation mask where each cell has a unique ID
    cell_coords : ndarray
        Array of cell coordinates
    """
    coords = adata.obsm['spatial']
    
    # Determine output shape
    if output_shape is None:
        max_x = int(np.ceil(coords[:, 0].max())) + 100
        max_y = int(np.ceil(coords[:, 1].max())) + 100
        output_shape = (max_y, max_x)
    
    # Create segmentation mask
    segmentation = np.zeros(output_shape, dtype=np.uint32)
    
    # Create circular masks for each cell
    y_indices, x_indices = np.ogrid[-cell_radius:cell_radius+1, -cell_radius:cell_radius+1]
    circle_mask = x_indices**2 + y_indices**2 <= cell_radius**2
    
    for i, (x, y) in enumerate(coords):
        cell_id = i + 1  # Cell IDs start at 1 (0 is background)
        
        # Convert to integer coordinates
        x_int = int(round(x))
        y_int = int(round(y))
        
        # Calculate bounds
        y_min = max(0, y_int - cell_radius)
        y_max = min(output_shape[0], y_int + cell_radius + 1)
        x_min = max(0, x_int - cell_radius)
        x_max = min(output_shape[1], x_int + cell_radius + 1)
        
        # Calculate mask bounds
        mask_y_min = max(0, cell_radius - y_int)
        mask_y_max = mask_y_min + (y_max - y_min)
        mask_x_min = max(0, cell_radius - x_int)
        mask_x_max = mask_x_min + (x_max - x_min)
        
        # Apply mask
        try:
            mask_region = circle_mask[mask_y_min:mask_y_max, mask_x_min:mask_x_max]
            segmentation[y_min:y_max, x_min:x_max][mask_region] = cell_id
        except (IndexError, ValueError):
            pass  # Skip cells at image boundaries
    
    return segmentation, coords


if adata_xenium is not None and 'spatial' in adata_xenium.obsm:
    print("Creating Xenium segmentation mask...")
    xenium_segmentation, xenium_coords = create_segmentation_from_coords(
        adata_xenium, 
        cell_radius=10  # Larger radius for visualization at 40x
    )
    
    print(f"\nSegmentation created:")
    print(f"  Shape: {xenium_segmentation.shape}")
    print(f"  Number of cells: {len(np.unique(xenium_segmentation)) - 1}")  # -1 for background
    print(f"  Coordinate range: X=[{xenium_coords[:, 0].min():.1f}, {xenium_coords[:, 0].max():.1f}], "
          f"Y=[{xenium_coords[:, 1].min():.1f}, {xenium_coords[:, 1].max():.1f}]")
    
    # Visualize original segmentation
    fig, axes = plt.subplots(1, 2, figsize=(15, 6))
    
    # Segmentation mask
    axes[0].imshow(xenium_segmentation > 0, cmap='gray')
    axes[0].set_title('Xenium Segmentation Mask (40x magnification)')
    axes[0].axis('off')
    
    # Cell coordinates
    axes[1].scatter(xenium_coords[:, 0], xenium_coords[:, 1], 
                    s=1, alpha=0.5, c=adata_xenium.obs['celltype'].cat.codes, cmap='tab20')
    axes[1].set_title('Xenium Cell Coordinates by Type')
    axes[1].set_xlabel('X')
    axes[1].set_ylabel('Y')
    axes[1].invert_yaxis()
    
    plt.tight_layout()
    plt.savefig(FIGURES_DIR / 'xenium_segmentation_original.png', dpi=300, bbox_inches='tight')
    plt.show()
else:
    print("Cannot create segmentation: Xenium data or spatial coordinates not available")
    xenium_segmentation = None

## 4. Resize Xenium Segmentation to Match Phenocycler Scale

Account for magnification difference: Xenium (40x) → Phenocycler (20x) requires 0.5x scaling.

In [ ]:
def resize_segmentation(segmentation, scale_factor, order=0):
    """
    Resize segmentation mask while preserving cell IDs.
    
    Parameters:
    -----------
    segmentation : ndarray
        Segmentation mask with integer cell IDs
    scale_factor : float
        Scaling factor (e.g., 0.5 for 2x downsampling)
    order : int
        Interpolation order (0=nearest neighbor for segmentation)
    
    Returns:
    --------
    resized : ndarray
        Resized segmentation mask
    """
    # Use nearest neighbor interpolation to preserve cell IDs
    resized = transform.rescale(
        segmentation,
        scale_factor,
        order=order,
        preserve_range=True,
        anti_aliasing=False
    )
    
    return resized.astype(segmentation.dtype)


def resize_coordinates(coords, scale_factor):
    """
    Resize spatial coordinates.
    
    Parameters:
    -----------
    coords : ndarray
        Array of coordinates (n_cells, 2)
    scale_factor : float
        Scaling factor
    
    Returns:
    --------
    resized_coords : ndarray
        Resized coordinates
    """
    return coords * scale_factor


if xenium_segmentation is not None:
    print(f"Resizing Xenium segmentation by factor {SCALE_FACTOR}...")
    print(f"Original shape: {xenium_segmentation.shape}")
    
    # Resize segmentation
    xenium_segmentation_resized = resize_segmentation(xenium_segmentation, SCALE_FACTOR)
    
    # Resize coordinates
    xenium_coords_resized = resize_coordinates(xenium_coords, SCALE_FACTOR)
    
    print(f"Resized shape: {xenium_segmentation_resized.shape}")
    print(f"Number of cells preserved: {len(np.unique(xenium_segmentation_resized)) - 1}")
    print(f"Resized coordinate range: X=[{xenium_coords_resized[:, 0].min():.1f}, {xenium_coords_resized[:, 0].max():.1f}], "
          f"Y=[{xenium_coords_resized[:, 1].min():.1f}, {xenium_coords_resized[:, 1].max():.1f}]")
    
    # Store resized coordinates in AnnData
    if adata_xenium is not None:
        adata_xenium.obsm['spatial_resized'] = xenium_coords_resized
    
    # Visualize resized segmentation
    fig, axes = plt.subplots(1, 2, figsize=(15, 6))
    
    # Original
    axes[0].imshow(xenium_segmentation > 0, cmap='gray')
    axes[0].set_title(f'Original Xenium Segmentation\n{xenium_segmentation.shape}')
    axes[0].axis('off')
    
    # Resized
    axes[1].imshow(xenium_segmentation_resized > 0, cmap='gray')
    axes[1].set_title(f'Resized Xenium Segmentation (scaled {SCALE_FACTOR}x)\n{xenium_segmentation_resized.shape}')
    axes[1].axis('off')
    
    plt.tight_layout()
    plt.savefig(FIGURES_DIR / 'xenium_segmentation_resized.png', dpi=300, bbox_inches='tight')
    plt.show()
else:
    print("Cannot resize: segmentation not available")
    xenium_segmentation_resized = None

## 5. Register Xenium Segmentation to Phenocycler Image

Perform image registration to align the resized Xenium segmentation with the Phenocycler image.

In [ ]:
def register_images(fixed_image, moving_image, upsample_factor=10):
    """
    Register moving image to fixed image using phase cross-correlation.
    
    Parameters:
    -----------
    fixed_image : ndarray
        Reference image (Phenocycler)
    moving_image : ndarray
        Image to register (Xenium segmentation)
    upsample_factor : int
        Upsampling factor for sub-pixel registration
    
    Returns:
    --------
    registered : ndarray
        Registered moving image
    shift : ndarray
        Calculated shift (y, x)
    """
    # Ensure images are same size (crop or pad if necessary)
    h_fixed, w_fixed = fixed_image.shape[:2]
    h_moving, w_moving = moving_image.shape[:2]
    
    # Pad or crop moving image to match fixed image
    if h_moving < h_fixed or w_moving < w_fixed:
        # Pad
        pad_h = max(0, h_fixed - h_moving)
        pad_w = max(0, w_fixed - w_moving)
        moving_padded = np.pad(moving_image, 
                               ((0, pad_h), (0, pad_w)), 
                               mode='constant')
    else:
        # Crop
        moving_padded = moving_image[:h_fixed, :w_fixed]
    
    # Convert to appropriate format for registration
    if fixed_image.ndim == 3:
        # Use first channel or create grayscale
        fixed_gray = np.mean(fixed_image, axis=-1) if fixed_image.shape[-1] == 3 else fixed_image[0]
    else:
        fixed_gray = fixed_image
    
    if moving_padded.ndim == 3:
        moving_gray = np.mean(moving_padded, axis=-1) if moving_padded.shape[-1] == 3 else moving_padded[0]
    else:
        moving_gray = moving_padded
    
    # Create binary mask from segmentation for registration
    moving_binary = (moving_gray > 0).astype(np.float32)
    
    # Normalize fixed image
    fixed_norm = (fixed_gray - fixed_gray.min()) / (fixed_gray.max() - fixed_gray.min() + 1e-8)
    
    try:
        # Calculate shift using phase cross-correlation
        shift, error, diffphase = phase_cross_correlation(
            fixed_norm, 
            moving_binary,
            upsample_factor=upsample_factor
        )
        
        print(f"Registration shift: {shift}")
        print(f"Registration error: {error:.4f}")
        
    except Exception as e:
        print(f"Registration failed: {e}")
        print("Using zero shift (no registration)")
        shift = np.array([0, 0])
    
    # Apply shift to moving image
    registered = ndimage.shift(moving_padded, shift, order=0)
    
    return registered, shift


def apply_shift_to_coords(coords, shift):
    """
    Apply registration shift to coordinates.
    
    Parameters:
    -----------
    coords : ndarray
        Coordinates (n_cells, 2) in (x, y) format
    shift : ndarray
        Shift vector (y, x) from registration
    
    Returns:
    --------
    shifted_coords : ndarray
        Registered coordinates
    """
    # Note: shift is (y, x) but coords are (x, y)
    shifted_coords = coords.copy()
    shifted_coords[:, 0] += shift[1]  # x shift
    shifted_coords[:, 1] += shift[0]  # y shift
    return shifted_coords


if phenocycler_image is not None and xenium_segmentation_resized is not None:
    print("Registering Xenium segmentation to Phenocycler image...")
    
    # Create reference image from Phenocycler
    if phenocycler_image.ndim == 3:
        # Use composite or first channel
        pheno_reference = create_composite_image(phenocycler_image)
    else:
        pheno_reference = phenocycler_image
    
    # Register
    xenium_segmentation_registered, registration_shift = register_images(
        pheno_reference,
        xenium_segmentation_resized
    )
    
    # Apply shift to coordinates
    xenium_coords_registered = apply_shift_to_coords(
        xenium_coords_resized,
        registration_shift
    )
    
    # Store registered coordinates in AnnData
    if adata_xenium is not None:
        adata_xenium.obsm['spatial_registered'] = xenium_coords_registered
    
    print(f"\nRegistration complete:")
    print(f"  Shift applied: {registration_shift}")
    print(f"  Registered coordinate range: X=[{xenium_coords_registered[:, 0].min():.1f}, {xenium_coords_registered[:, 0].max():.1f}], "
          f"Y=[{xenium_coords_registered[:, 1].min():.1f}, {xenium_coords_registered[:, 1].max():.1f}]")
    
else:
    print("Cannot register: Phenocycler image or Xenium segmentation not available")
    xenium_segmentation_registered = None
    registration_shift = None

## 6. Visualize Registration Results

Overlay the registered Xenium segmentation on the Phenocycler image to verify alignment.

In [ ]:
def create_overlay(background, overlay_mask, alpha=0.5, overlay_color=[1, 0, 0]):
    """
    Create overlay visualization.
    
    Parameters:
    -----------
    background : ndarray
        Background image (can be grayscale or RGB)
    overlay_mask : ndarray
        Binary or segmentation mask
    alpha : float
        Transparency of overlay
    overlay_color : list
        RGB color for overlay
    
    Returns:
    --------
    overlay : ndarray
        Overlay image
    """
    # Normalize background to 0-1
    if background.max() > 1:
        background_norm = background.astype(np.float32) / background.max()
    else:
        background_norm = background.astype(np.float32)
    
    # Convert grayscale to RGB if needed
    if background_norm.ndim == 2:
        background_rgb = np.stack([background_norm] * 3, axis=-1)
    else:
        background_rgb = background_norm
    
    # Create colored overlay
    overlay = background_rgb.copy()
    mask_binary = (overlay_mask > 0)
    
    for i in range(3):
        overlay[:, :, i] = np.where(
            mask_binary,
            alpha * overlay_color[i] + (1 - alpha) * background_rgb[:, :, i],
            background_rgb[:, :, i]
        )
    
    return overlay


if phenocycler_image is not None and xenium_segmentation_registered is not None:
    print("Creating overlay visualization...")
    
    # Create composite from Phenocycler
    pheno_composite = create_composite_image(phenocycler_image)
    
    # Ensure same size
    h, w = pheno_composite.shape[:2]
    seg_display = xenium_segmentation_registered[:h, :w]
    
    # Create overlay
    overlay_image = create_overlay(
        pheno_composite,
        seg_display,
        alpha=0.4,
        overlay_color=[1, 1, 0]  # Yellow
    )
    
    # Visualize
    fig, axes = plt.subplots(1, 3, figsize=(20, 6))
    
    # Phenocycler
    axes[0].imshow(pheno_composite)
    axes[0].set_title('Phenocycler Image (20x)')
    axes[0].axis('off')
    
    # Xenium segmentation
    axes[1].imshow(seg_display > 0, cmap='gray')
    axes[1].set_title('Registered Xenium Segmentation')
    axes[1].axis('off')
    
    # Overlay
    axes[2].imshow(overlay_image)
    axes[2].set_title('Overlay (Phenocycler + Xenium Segmentation)')
    axes[2].axis('off')
    
    plt.tight_layout()
    plt.savefig(FIGURES_DIR / 'registration_overlay.png', dpi=300, bbox_inches='tight')
    plt.show()
    
    # Plot cell coordinates on Phenocycler
    if adata_xenium is not None and 'spatial_registered' in adata_xenium.obsm:
        fig, ax = plt.subplots(1, 1, figsize=(12, 10))
        
        # Background
        ax.imshow(pheno_composite, alpha=0.6)
        
        # Cell coordinates colored by type
        scatter = ax.scatter(
            xenium_coords_registered[:, 0],
            xenium_coords_registered[:, 1],
            c=adata_xenium.obs['celltype'].cat.codes,
            s=2,
            alpha=0.7,
            cmap='tab20'
        )
        
        ax.set_title('Registered Xenium Cells on Phenocycler Image')
        ax.set_xlabel('X')
        ax.set_ylabel('Y')
        
        # Legend
        handles, labels = scatter.legend_elements()
        cell_types = adata_xenium.obs['celltype'].cat.categories
        if len(cell_types) <= 20:  # Only show legend if not too many types
            ax.legend(handles, cell_types, bbox_to_anchor=(1.05, 1), loc='upper left', 
                     fontsize=8, markerscale=2)
        
        plt.tight_layout()
        plt.savefig(FIGURES_DIR / 'registered_cells_overlay.png', dpi=300, bbox_inches='tight')
        plt.show()
else:
    print("Cannot create overlay: required data not available")

## 7. Quality Assessment

Assess the quality of registration and integration.

In [ ]:
if adata_xenium is not None and xenium_segmentation_registered is not None:
    # Calculate registration statistics
    print("Registration Quality Assessment:")
    print("=" * 50)
    
    # Original vs registered coordinate ranges
    if 'spatial' in adata_xenium.obsm:
        orig_coords = adata_xenium.obsm['spatial']
        print(f"\nOriginal Xenium coordinates (40x):")
        print(f"  X range: [{orig_coords[:, 0].min():.1f}, {orig_coords[:, 0].max():.1f}]")
        print(f"  Y range: [{orig_coords[:, 1].min():.1f}, {orig_coords[:, 1].max():.1f}]")
    
    if 'spatial_resized' in adata_xenium.obsm:
        resized_coords = adata_xenium.obsm['spatial_resized']
        print(f"\nResized Xenium coordinates (scaled to 20x):")
        print(f"  X range: [{resized_coords[:, 0].min():.1f}, {resized_coords[:, 0].max():.1f}]")
        print(f"  Y range: [{resized_coords[:, 1].min():.1f}, {resized_coords[:, 1].max():.1f}]")
        print(f"  Scale factor applied: {SCALE_FACTOR}")
    
    if 'spatial_registered' in adata_xenium.obsm:
        reg_coords = adata_xenium.obsm['spatial_registered']
        print(f"\nRegistered Xenium coordinates (aligned to Phenocycler):")
        print(f"  X range: [{reg_coords[:, 0].min():.1f}, {reg_coords[:, 0].max():.1f}]")
        print(f"  Y range: [{reg_coords[:, 1].min():.1f}, {reg_coords[:, 1].max():.1f}]")
        if registration_shift is not None:
            print(f"  Registration shift: {registration_shift}")
    
    if phenocycler_image is not None:
        print(f"\nPhenocycler image dimensions:")
        print(f"  Shape: {phenocycler_image.shape}")
        if phenocycler_image.ndim == 3:
            print(f"  Height x Width: {phenocycler_image.shape[1]} x {phenocycler_image.shape[2]}")
        else:
            print(f"  Height x Width: {phenocycler_image.shape[0]} x {phenocycler_image.shape[1]}")
    
    # Cell type distribution
    print(f"\nCell type distribution:")
    cell_type_counts = adata_xenium.obs['celltype'].value_counts()
    for ct, count in cell_type_counts.head(10).items():
        print(f"  {ct}: {count}")
    if len(cell_type_counts) > 10:
        print(f"  ... and {len(cell_type_counts) - 10} more types")
    
else:
    print("Cannot assess quality: required data not available")

## 8. Extract Phenocycler Intensities at Cell Locations

Sample Phenocycler channel intensities at registered Xenium cell locations.

In [ ]:
def extract_intensities_at_coords(image, coords, radius=2):
    """
    Extract image intensities at specified coordinates.
    
    Parameters:
    -----------
    image : ndarray
        Multi-channel image (channels, height, width) or (height, width)
    coords : ndarray
        Cell coordinates (n_cells, 2) in (x, y) format
    radius : int
        Radius for local averaging
    
    Returns:
    --------
    intensities : ndarray
        Intensity values (n_cells, n_channels)
    """
    if image.ndim == 2:
        # Single channel
        image = image[np.newaxis, :, :]
    
    n_channels = image.shape[0]
    n_cells = len(coords)
    intensities = np.zeros((n_cells, n_channels))
    
    h, w = image.shape[1:]
    
    for i, (x, y) in enumerate(coords):
        # Convert to integer
        x_int = int(round(x))
        y_int = int(round(y))
        
        # Check bounds
        if 0 <= x_int < w and 0 <= y_int < h:
            # Extract local region
            y_min = max(0, y_int - radius)
            y_max = min(h, y_int + radius + 1)
            x_min = max(0, x_int - radius)
            x_max = min(w, x_int + radius + 1)
            
            # Average over local region
            for ch in range(n_channels):
                local_region = image[ch, y_min:y_max, x_min:x_max]
                intensities[i, ch] = np.mean(local_region)
    
    return intensities


if phenocycler_image is not None and adata_xenium is not None and 'spatial_registered' in adata_xenium.obsm:
    print("Extracting Phenocycler intensities at Xenium cell locations...")
    
    # Extract intensities
    pheno_intensities = extract_intensities_at_coords(
        phenocycler_image,
        adata_xenium.obsm['spatial_registered'],
        radius=3
    )
    
    print(f"\nExtracted intensities:")
    print(f"  Shape: {pheno_intensities.shape}")
    print(f"  Channels: {pheno_intensities.shape[1]}")
    print(f"  Cells: {pheno_intensities.shape[0]}")
    
    # Add to AnnData as a layer or obsm
    adata_xenium.obsm['phenocycler_intensities'] = pheno_intensities
    
    # Also add channel-wise to obs for easy access
    for ch_idx in range(pheno_intensities.shape[1]):
        adata_xenium.obs[f'pheno_ch{ch_idx}'] = pheno_intensities[:, ch_idx]
    
    # Visualize intensity distributions
    fig, axes = plt.subplots(2, 3, figsize=(15, 10))
    axes = axes.flatten()
    
    n_to_plot = min(6, pheno_intensities.shape[1])
    for ch_idx in range(n_to_plot):
        axes[ch_idx].hist(pheno_intensities[:, ch_idx], bins=50, edgecolor='black')
        axes[ch_idx].set_title(f'Channel {ch_idx} Intensity Distribution')
        axes[ch_idx].set_xlabel('Intensity')
        axes[ch_idx].set_ylabel('Count')
    
    # Hide unused subplots
    for idx in range(n_to_plot, 6):
        axes[idx].axis('off')
    
    plt.tight_layout()
    plt.savefig(FIGURES_DIR / 'phenocycler_intensity_distributions.png', dpi=300, bbox_inches='tight')
    plt.show()
    
else:
    print("Cannot extract intensities: required data not available")

## 9. Save Integrated Data

In [ ]:
if adata_xenium is not None:
    # Add integration metadata
    adata_xenium.uns['integration_params'] = {
        'xenium_magnification': XENIUM_MAGNIFICATION,
        'phenocycler_magnification': PHENOCYCLER_MAGNIFICATION,
        'scale_factor': SCALE_FACTOR,
        'registration_shift': list(registration_shift) if registration_shift is not None else None,
        'phenocycler_shape': phenocycler_image.shape if phenocycler_image is not None else None
    }
    
    # Save integrated data
    output_file = OUTPUT_DIR / f"{SAMPLE_NAME}_integrated.h5ad"
    adata_xenium.write_h5ad(output_file)
    
    print(f"\nIntegrated data saved to: {output_file}")
    print(f"\nSaved coordinate sets:")
    for key in adata_xenium.obsm.keys():
        if 'spatial' in key:
            print(f"  - {key}: {adata_xenium.obsm[key].shape}")
    
    print(f"\nSaved intensity data:")
    if 'phenocycler_intensities' in adata_xenium.obsm:
        print(f"  - phenocycler_intensities: {adata_xenium.obsm['phenocycler_intensities'].shape}")
    
    # Save registered segmentation
    if xenium_segmentation_registered is not None:
        seg_file = OUTPUT_DIR / f"{SAMPLE_NAME}_xenium_segmentation_registered.tif"
        tifffile.imwrite(seg_file, xenium_segmentation_registered.astype(np.uint32))
        print(f"\nRegistered segmentation saved to: {seg_file}")
    
    print("\n" + "="*50)
    print("Integration complete!")
    print("="*50)
    print(f"\nSummary:")
    print(f"  - Xenium cells: {adata_xenium.n_obs}")
    print(f"  - Cell types: {adata_xenium.obs['celltype'].nunique()}")
    print(f"  - Spatial coordinate sets: {len([k for k in adata_xenium.obsm.keys() if 'spatial' in k])}")
    if 'phenocycler_intensities' in adata_xenium.obsm:
        print(f"  - Phenocycler channels integrated: {adata_xenium.obsm['phenocycler_intensities'].shape[1]}")
else:
    print("No data to save")

## 10. Summary and Next Steps

### What was done:
1. ✅ Loaded Xenium spatial transcriptomics data
2. ✅ Read raw Phenocycler data from .qptiff format
3. ✅ Created Xenium segmentation mask from cell coordinates
4. ✅ Accounted for magnification difference (40x → 20x)
5. ✅ Resized Xenium segmentation to match Phenocycler scale
6. ✅ Registered resized segmentation to Phenocycler image
7. ✅ Extracted Phenocycler intensities at Xenium cell locations
8. ✅ Saved integrated multi-modal dataset

### Coordinate sets in integrated data:
- `spatial`: Original Xenium coordinates (40x magnification)
- `spatial_resized`: Coordinates scaled to match Phenocycler (20x magnification)
- `spatial_registered`: Final registered coordinates aligned to Phenocycler image

### Next steps:
1. **Quality control**: Verify registration quality visually
2. **Cross-modal analysis**: Compare transcriptomics (Xenium) with proteomics (Phenocycler)
3. **Spatial analysis**: Perform joint spatial analysis on integrated data
4. **Cell type validation**: Compare cell types identified by RNA vs protein markers
5. **Advanced integration**: Use methods like totalVI for joint modeling

### Key parameters to adjust:
- `SCALE_FACTOR`: Adjust if using different magnifications
- `cell_radius`: Cell size in segmentation mask
- `upsample_factor`: Sub-pixel registration precision
- `radius` in intensity extraction: Size of local region for sampling